# `08_task_d_extraction.ipynb` — Target-String Generation & Deterministic Span Alignment (`D1`, `D2`)

## 1. Architectural Separation: LLM Extracts Strings, Software Computes Spans
Task D (Opinion Target Expression / OTE extraction) requires identifying the exact opinion target phrase for each explicit opinion, along with its aspect category and polarity.

### Why Never Train the LLM on Character Offsets?
In raw XML, an opinion looks like `<Opinion category="Zugfahrt#Haupt" from="11" to="23" target="Regionalbahn" polarity="negative"/>`. Asking an LLM to generate `"from": 11, "to": 23` fails because:
- Tokenizers group characters into variable-length subwords; the model has no internal character counter.
- Off-by-one character shifts immediately produce `0.0` exact-span F1.

### Our Two-Stage Task D Architecture
1. **Stage 1 — Generative String Extraction (LLM):**
   The LLM emits verbatim copied target substrings in text order (using `' * '` when a German verbal bracket or multi-word target is split across non-adjacent words, e.g., `"fiel * aus"`):
   ```json
   {"targets": [{"target": "ICE", "category": "Zugfahrt", "polarity": "positive"}, {"target": "fiel * aus", "category": "Zugfahrt", "polarity": "negative"}]}
   ```
2. **Stage 2 — Deterministic Character Alignment (`src/germeval_llm/alignment.py`):**
   - **Unique Contiguous Match (`unique_contiguous`):** `text.find(target)` finds a single exact match $\rightarrow$ returns `[start, end)`.
   - **Ambiguous Repeated Match (`ambiguous_contiguous`):** When the same word (e.g., `"ICE"`) appears multiple times in a post, the aligner assigns occurrences left-to-right (`leftmost_unused` cursor per `(target, category)`).
   - **Discontinuous `' * '` Match (`discontinuous_star`):** Splits `"A * B"` into ordered segments `["A", "B"]`, matches them sequentially in the document, and returns the bounding span `[start_A, end_B)` matching the GermEval XML convention.
   - **Unaligned / Hallucinated (`unaligned`):** Flagged as `start_char=None, end_char=None`; counted as a false positive in our pure-Python scorer and excluded from XML export so `EvaluationScript.jar` never crashes.

In [ ]:
from pathlib import Path
import os, sys, json
from IPython.display import Image, display, Markdown
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
    os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from germeval_llm.convert import ensure_derived_datasets_ready
ensure_derived_datasets_ready()
print("Repository initialized:", ROOT.name)


In [ ]:
td = json.loads(Path('results/metrics/task_d_summary.json').read_text())
rows = []
for run_id, sp_dict in td.items():
    for sp, m in sp_dict.items():
        rows.append({
            'run_id': run_id,
            'split': sp,
            'string_cat_pol_f1': round(m['string_multiset_category_polarity']['f1'], 4),
            'string_cat_only_f1': round(m['string_multiset_category_only']['f1'], 4),
            'span_exact_python_f1': round(m['span_explicit_exact']['f1'], 4),
            'span_overlap_python_f1': round(m['span_explicit_overlap']['f1'], 4),
            'official_jar_exact_f1': round(m['official_jar_explicit_gold_ote']['exact_micro_f1'], 4),
            'official_jar_overlap_f1': round(m['official_jar_explicit_gold_ote']['overlap_micro_f1'], 4),
            'unique_aligned_rate': round(m['alignment_breakdown']['rates'].get('unique_contiguous', 0.0), 3),
            'ambiguous_rate': round(m['alignment_breakdown']['rates'].get('ambiguous_contiguous', 0.0), 3),
            'discontinuous_star_rate': round(m['alignment_breakdown']['rates'].get('discontinuous_star', 0.0), 3),
        })
display(pd.DataFrame(rows))

### Hands-on Classroom Demo: Interactive Span Aligner & Pure-Python Task D Scorer
Test `align_targets_to_text` on a sentence with repeated nouns (`ICE`), a discontinuous verbal bracket (`fiel * aus`), and a hallucinated target (`Bordbistro`):

In [ ]:
from germeval_llm.alignment import align_targets_to_text
from germeval_llm.schemas import TaskDTargetItem

demo_text = 'Der ICE war pünktlich, aber der zweite ICE fiel wegen Schnee komplett aus.'
demo_preds = [
    TaskDTargetItem(target='ICE', category='Zugfahrt', polarity='positive'),
    TaskDTargetItem(target='ICE', category='Zugfahrt', polarity='negative'),
    TaskDTargetItem(target='fiel * aus', category='Zugfahrt', polarity='negative'),
    TaskDTargetItem(target='Bordbistro', category='Gastronomisches_Angebot', polarity='negative'),
]
for sp in align_targets_to_text(demo_text, demo_preds):
    slice_str = demo_text[sp.start_char:sp.end_char] if sp.start_char is not None else None
    print(f'{sp.target:12s} | status={sp.alignment_status:20s} | span=({str(sp.start_char):>4s}, {str(sp.end_char):>4s}) -> {slice_str!r}')

### Key Takeaways
- Prompting (`D1`) struggles with the GermEval discontinuous `' * '` convention (e.g., German verbal brackets), whereas QLoRA (`D2`) learns to emit `' * '`-joined targets reliably (**+19.7 F1 points** on `test_syn`).
- Both String-level Multiset F1 and Explicit-Span Exact/Overlap F1 are computed in **100% pure Python** (`evaluate_task_d_predictions`), while also reconciling with `EvaluationScript.jar OTE`.